# Day 4 — Census & Demand Estimation

**Estimated time:** 2:00 PM – 5:00 PM  •  **Prereq:** Day 3 distances

Today you build the model that answers the question.

Three sections:

1. **Census data.** Who depends on MARTA to get to work? We pull two
   American Community Survey tables to find out.
2. **The 3×3 demand grid.** Match-day rail demand vs. match-day rail
   capacity, across nine combinations of assumptions.
3. **Anchors and cross-checks.** Super Bowl LIII and a Georgia Tech
   regression both give us reality checks on our numbers.

## Learning objectives

- [ ] Read two Census ACS tables and compute percentages.
- [ ] Build a parametric scenario grid (three mode shares × three
      train capacities).
- [ ] Interpret a 3×3 gap matrix — where are we surplus, where deficit?
- [ ] Cross-check a model against an independent estimate.
- [ ] Save `data/checkpoints/day4_output_demand_estimates.csv`.

## Tiered exercises

Today's exercises are marked 🟢 (basic), 🔵 (standard), ⬛ (stretch).
**Everyone should finish the 🟢 + 🔵 exercises.** ⬛ is optional — pick
it up if you have time at the end.


## 1. Setup


In [ ]:
# Setup — run this first.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'data' / 'raw').is_dir():
    ROOT = ROOT.parent
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
CHECKPOINTS = ROOT / 'data' / 'checkpoints'

MARTA_NTD_ID = 40022
MODE_NAMES = {'HR': 'Heavy Rail', 'MB': 'Bus', 'SR': 'Streetcar',
              'DR': 'Demand Response'}
STADIUM_ADJACENT = ['sec district', 'vine city']
STADIUM_AREA = STADIUM_ADJACENT + ['five points']

# World Cup demand model (Day 4). Canonical values; sources cited in
# the Day 4 notebook narrative. Changing any value here propagates
# through Day 4 and Day 5 with no duplicate literals to update.
STADIUM_CAPACITY = 75_000
MATCH_DAY_TRAINS_PER_HOUR = 24
ARRIVAL_WINDOW = 2.5
MODE_SHARE = {'low': 0.15, 'central': 0.22, 'high': 0.35}
TRAIN_CAP = {'seated_only': 232, 'service_standard': 400, 'full_crush': 750}
MS_ORDER = ('low', 'central', 'high')
CAP_ORDER = ('seated_only', 'service_standard', 'full_crush')
# GT regression (Santanam et al. 2021, arXiv:2106.05359; MAPE 11.7%).
GT_INTERCEPT = -1201
GT_SLOPE = 0.1739

print(f'✅ Imports ready. Working from {ROOT}')


In [ ]:
# Reload today's prerequisites from Day 2 + Day 3 checkpoints.
ridership = pd.read_csv(
    CHECKPOINTS / 'day2_output_cleaned_ridership.csv', parse_dates=['date']
)
station_freq = pd.read_csv(CHECKPOINTS / 'day2_output_station_freq_clean.csv')
eda_summary = pd.read_csv(CHECKPOINTS / 'day3_output_eda_summary.csv')
print(f'ridership rows: {len(ridership):,}')
print(f'stations:       {len(station_freq)}')
print(f'eda_summary rows: {len(eda_summary)}')


## 2. Section 1 — Census: who depends on MARTA?

Before we model World Cup demand, we anchor the analysis in *who
already rides*. The Census Bureau's American Community Survey publishes
journey-to-work data at the Census tract level. We already pulled two
tables for Fulton + DeKalb counties:

- **B08301** — Means of Transportation to Work. We care about
  `public_transit` as a share of `total_workers`.
- **B08141** — Means of Transportation by Vehicles Available. We care
  about `workers_zero_vehicle_hh` — a proxy for *transit-dependent*
  workers.

Both files live in `data/raw/`. The column names come straight from
the Census API (cryptic), so our first job is to rename them.


> **Why these two ACS tables — what the equity variable adds**
>
> A high `pct_transit` could just mean "transit is a good option in
> this neighborhood." A high `pct_workers_in_zero_vehicle_hh` means
> "transit is the *only* option" — those are the riders with no car
> to fall back on if the trains get unusable. If the World Cup makes
> daily rail unbearable for a month, B08141 tells you whose commutes
> get hurt. That's the equity story the briefing should at least
> acknowledge — even if the model doesn't explicitly include it.


### 🟢 Exercise 1 — Rename and clean

Load `acs_b08301_fulton_dekalb.csv`, rename the cryptic columns to
human names, and coerce them to numeric (they arrive as strings).


In [ ]:
# 🟢 Your Turn! Load the file and rename these three columns:
#   B08301_001E -> total_workers
#   B08301_003E -> drove_alone
#   B08301_010E -> public_transit
b08301 = pd.read_csv(???)
b08301 = b08301.rename(columns={
    ???: 'total_workers',
    ???: 'drove_alone',
    ???: 'public_transit',
})
# Now coerce each of the three renamed columns to numeric:
for c in ???:
    b08301[c] = pd.to_numeric(b08301[c], errors='coerce')
print(f'Tracts: {len(b08301)}')
b08301[['NAME', 'total_workers', 'drove_alone', 'public_transit']].head()


### 🟢 Exercise 2 — percent transit commuters

Compute `pct_transit = public_transit / total_workers * 100` as a new
column. Then print the **median** and **mean** percentages across the
county.


In [ ]:
# 🟢 Your Turn!
b08301['pct_transit'] = ???
print(f"Median: {???:.1f}%,  Mean: {???:.1f}%")


<details><summary>💡 Click for solution</summary>

```python
b08301['pct_transit'] = (
    b08301['public_transit'] / b08301['total_workers'] * 100
).round(1)
```

The **median is ~3-4%** across Fulton + DeKalb — low because most
tracts are suburban, car-dependent neighborhoods. The **top 20 tracts
reach 25-40%** — mostly in-town dense neighborhoods around Five Points,
Midtown, and the West End.
</details>


### 🔵 Exercise 3 — B08141, the equity table

Load `acs_b08141_fulton_dekalb.csv` and compute
`pct_workers_in_zero_vehicle_hh`. This is the **equity** variable — it
measures workers who have *no car alternative* if MARTA fails them.
Column renames:

- `B08141_001E → total_workers_b08141`
- `B08141_002E → workers_zero_vehicle_hh`


In [ ]:
# 🔵 Your Turn!
b08141 = pd.read_csv(???)
b08141 = b08141.rename(columns={
    ???: 'total_workers_b08141',
    ???: 'workers_zero_vehicle_hh',
})
for c in ['total_workers_b08141', 'workers_zero_vehicle_hh']:
    b08141[c] = pd.to_numeric(b08141[c], errors='coerce')
b08141['pct_workers_in_zero_vehicle_hh'] = ???
print(
    f"Workers in 0-vehicle HH — median: "
    f"{b08141['pct_workers_in_zero_vehicle_hh'].median():.1f}%, "
    f"mean: {b08141['pct_workers_in_zero_vehicle_hh'].mean():.1f}%"
)


### ⬛ Exercise 4 — Top 20 tracts bar chart (optional)

Build a horizontal bar chart of the top 20 Census tracts by
`pct_transit`. The chart should:

- Use MARTA blue (`#0075B2`) for bars
- Show a dashed line at the county median
- Clean up the `NAME` column by stripping the
  `"; Fulton County; Georgia"` and `"; DeKalb County; Georgia"`
  suffixes before labeling the y-axis.

Save the figure as `data/checkpoints/census_transit_dependence.png`.


In [ ]:
# ⬛ Stretch. Skip if you're short on time — the 3×3 grid is more
# important.
top20 = b08301.nlargest(20, 'pct_transit').copy()
top20['tract_label'] = (
    top20['NAME']
    .str.replace(r'; Fulton County; Georgia$', '', regex=True)
    .str.replace(r'; DeKalb County; Georgia$', '', regex=True)
    .str.replace('Census Tract ', 'Tract ', regex=False)
)

fig, ax = plt.subplots(figsize=(10, 8))
# 🎯 Your Turn! Finish the chart. One barh() call, one axvline at the
# county median, labels from top20['tract_label'], save to
# CHECKPOINTS / 'census_transit_dependence.png'.
???


## 🔄 Recovery point — end of Section 1

If the Census work gave you trouble, run the next cell to jump
cleanly to Section 2.


In [ ]:
# Recovery: you only need the ridership table from here onward.
ridership = pd.read_csv(
    CHECKPOINTS / 'day2_output_cleaned_ridership.csv', parse_dates=['date']
)
print('✅ Recovery loaded.')


## 3. Section 2 — The 3×3 demand grid

### The logic

- **Demand.** Stadium capacity × transit mode share. The stadium's
  World Cup configuration seats **75,000** (the MLS/Falcons seated
  setup is 71,000 — FIFA reconfigures). Mode share — what fraction of
  fans ride MARTA — is **not known**. We use three anchored scenarios:
  - **Low 15%** — Georgia Tech study's regular-season observation at
    this stadium (Santanam et al. 2021, arXiv:2106.05359).
  - **Central 22%** — mega-event uplift over regular season.
  - **High 35%** — Super Bowl LIII context, where 155,000 system-wide
    rail riders implied mode shares well above regular-season.

- **Supply.** Match-day trains per hour per direction at the two
  stadium-adjacent stations × per-train capacity × 2.5-hour pre-match
  window. MARTA's March 2026 Board commitment: **5-minute headways per
  line = 24 trains/hour/direction** across the two stadium lines.
  Train capacity uses three Stadler CQ400 spec values:
  - **Seated only: 232**
  - **MARTA service standard: ~400** (170% of seated)
  - **Full crush: 750** (6 passengers per square meter)

- **Baseline commuters are *not* subtracted.** PM commuters go
  outbound from downtown; match-day fans go inbound — they use
  different track directions. 5 of 8 matches also kick off at noon,
  outside the PM commuter window entirely.

3 mode shares × 3 train capacities = **9 scenarios**. We'll compute
all nine and look at the gap.


> **Why both dimensions matter — what the tipping cell tells the reader**
>
> The 3×3 grid is the load-bearing artifact of the entire week, and
> its structure has a deliberate analytical purpose: *no single
> assumption flips the verdict*. Two students with defensible but
> different assumptions can both be right and reach different
> conclusions. The closest-to-zero cell — High mode share × Service
> standard, around −2,250 riders — is what the briefing's
> recommendation should actually anchor on. A change in either
> dimension swings that cell from deficit to surplus.


In [ ]:
# Model parameters live in setup/SETUP_CODE as module constants so the
# same values flow through Day 4 and Day 5 with no drift. If you want
# to play with a different capacity or mode share, override here and
# re-run — your override will shadow the shared value for this kernel.
print(f'Stadium capacity: {STADIUM_CAPACITY:,}')
print(f'Match-day trains/hr per direction: {MATCH_DAY_TRAINS_PER_HOUR}')
print(f'Arrival window: {ARRIVAL_WINDOW} hours')
print(f'Mode-share scenarios: {MODE_SHARE}')
print(f'Train capacity scenarios: {TRAIN_CAP}')


### 🔵 Exercise 5 — Build the 3×3 gap table

For each (mode_share, train_cap) combination, compute:

- `demand = STADIUM_CAPACITY * mode_share`
- `capacity = MATCH_DAY_TRAINS_PER_HOUR * train_cap * ARRIVAL_WINDOW`
- `gap = capacity - demand`
- `verdict = 'SURPLUS' if gap > 0 else 'DEFICIT'`

Collect the results as a list of dicts and build a DataFrame.


In [ ]:
# 🔵 Your Turn!
results = []
for ms_name, ms_val in MODE_SHARE.items():
    demand = ???
    for cap_name, cap_val in TRAIN_CAP.items():
        capacity = ???
        gap = ???
        results.append(dict(
            mode_share_scenario=ms_name,
            capacity_scenario=cap_name,
            demand_riders=int(demand),
            rail_capacity_riders=int(capacity),
            gap_riders=int(gap),
            verdict='SURPLUS' if gap > 0 else 'DEFICIT',
        ))

gap_df = pd.DataFrame(results)
pivot = gap_df.pivot_table(
    index='mode_share_scenario', columns='capacity_scenario',
    values='gap_riders', aggfunc='first',
)[list(CAP_ORDER)]

print('=== 3×3 Demand vs. Capacity Gap (riders) ===')
print('Positive = surplus, Negative = deficit')
print(pivot.to_string())


### ✅ Checkpoint — save the grid


In [ ]:
gap_df.to_csv(CHECKPOINTS / 'day4_output_demand_estimates.csv', index=False)
n_s = (gap_df['verdict'] == 'SURPLUS').sum()
n_d = (gap_df['verdict'] == 'DEFICIT').sum()
assert n_s + n_d == 9, f'Expected 9 scenarios; got {n_s + n_d}.'
assert n_s > 0 and n_d > 0, (
    'All scenarios agree — that would mean the analysis has no tension. '
    'Check STADIUM_CAPACITY and MATCH_DAY_TRAINS_PER_HOUR.'
)
print(f'✅ {n_s}/9 surplus, {n_d}/9 deficit. day4_output_demand_estimates.csv written.')


### 🔵 Exercise 6 — Plot the signature heatmap

This is the chart that answers the question at a glance. Diverging
colormap: red = deficit, green = surplus, annotation in each cell.


In [ ]:
# 🔵 Your Turn — finish the heatmap.
# MS_ORDER and CAP_ORDER are canonical display orders from setup.
ms_labels  = ['Low (15%)', 'Central (22%)', 'High (35%)']
cap_labels = ['Seated Only\n(232/train)',
              'Service Standard\n(~400/train)',
              'Full Crush\n(750/train)']

grid = pivot.reindex(list(MS_ORDER))[list(CAP_ORDER)].values.astype(float)
max_abs = max(abs(grid.min()), abs(grid.max()))

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(grid, cmap='RdYlGn', vmin=-max_abs, vmax=max_abs, aspect='auto')

# 🎯 Annotate each cell with the gap value and verdict. Use white text
# when |value| is large (>60% of max_abs), black otherwise.
for i in range(3):
    for j in range(3):
        val = int(grid[i, j])
        verdict = 'SURPLUS' if val > 0 else 'DEFICIT'
        color = ???   # white if large, black otherwise
        ax.text(j, i, f'{val:+,d}\n{verdict}', ha='center', va='center',
                fontsize=10, fontweight='bold', color=color)

ax.set_xticks(range(3)); ax.set_xticklabels(cap_labels, fontsize=9)
ax.set_yticks(range(3)); ax.set_yticklabels(ms_labels, fontsize=10)
ax.set_xlabel('Train Capacity Scenario')
ax.set_ylabel('Transit Mode Share Scenario')
ax.set_title('Can MARTA Handle the World Cup?\n'
             'Gap = Rail Capacity − Stadium Demand',
             fontsize=11, fontweight='bold')
fig.colorbar(im, ax=ax, shrink=0.8, label='Gap (riders)')
fig.tight_layout()
fig.savefig(CHECKPOINTS / 'gap_heatmap.png', dpi=150)
plt.show()


### The verdict

**6 of 9 scenarios show surplus. 3 show deficit.** The answer depends
on both *how many fans take the train* and *how full the trains
actually run*.

- At **seated-only** capacity: surplus at low mode share, deficit at
  central and high.
- At **service standard (~400)**: surplus at low and central; **-2,250
  riders at high** — the single tightest cell, the tipping point.
- At **full crush**: surplus everywhere.

The structural point: *both dimensions matter*. No single assumption
dominates. Two students with different assumptions can both be right
and reach different conclusions.


## 🔄 Recovery point — end of Section 2

If the heatmap failed, run the next cell to reload `gap_df` from the
checkpoint and continue with Section 3.


In [ ]:
gap_df = pd.read_csv(CHECKPOINTS / 'day4_output_demand_estimates.csv')
print('✅ Recovery loaded.')


## 4. Section 3 — Anchors and cross-checks

### Georgia Tech regression

Santanam et al. (2021, [arXiv:2106.05359](https://arxiv.org/abs/2106.05359))
fitted a linear regression across 134 event days at MBS and State Farm
Arena using MARTA Breeze Card fare data:

> **Post-game ridership = −1,201 + 0.1739 × Attendance**
> MAPE: **11.7%** (standalone LR; combined LR+RF model: 11.3%)

For a 70,000-seat event — the attendance the GT data reflects — this
predicts ~10,972 post-game rail riders. Our central scenario
(22% × 75,000 = 16,500) sits about **50% above** that baseline —
consistent with mega-event uplift.


> **Why we cross-check against an independent estimate**
>
> Our 3×3 grid uses three published anchors (Stadler CQ400 specs,
> MARTA's Board headway commitment, Census-anchored mode share).
> All from the same project. The Georgia Tech regression
> (Santanam et al. 2021, MAPE 11.7%) is independent — different
> authors, different method, different data (134 event days at MBS
> + State Farm Arena). When two independent estimates agree within a
> ~50% mega-event uplift, the model gets a lot more credible. When
> they disagree wildly, *that* is your headline.


### 🟢 Exercise 7 — cross-check the central scenario


In [ ]:
# 🟢 Your Turn!
# 1. Compute the GT regression prediction at attendance = 70_000 using
#    the module constants GT_INTERCEPT and GT_SLOPE (defined in setup).
# 2. Compute our central scenario demand: STADIUM_CAPACITY × central
#    mode share (both from the setup cell).
# 3. Express our central as a percent uplift over the GT baseline.
gt_baseline = ???
central_demand = ???
uplift_pct = ???
print(f'GT baseline (70k event):    {gt_baseline:>8,.0f} riders')
print(f'Our central (22% of 75k):   {central_demand:>8,.0f} riders')
print(f'Uplift over GT baseline:    {uplift_pct:+.0f}%')


### Super Bowl LIII anchor

On Super Bowl Sunday 2019 MARTA carried **155,000 rail riders**
system-wide and cleared stadium crowds within **90 minutes** of the
final whistle. Same venue, same agency, proven operational capacity.

With ~80,000 baseline Sunday riders, the Super Bowl generated about
**75,000 incremental riders** — the single strongest piece of evidence
that MARTA can handle a 70k-seat mega-event at this stadium when
service is deployed properly.

*Source: MARTA press release, Feb 4, 2019; confirmed by Progressive
Railroading, Metro Magazine, 11Alive, AJC.*

### What we did NOT model

- **Fleet mix.** CQ400 trainsets are only just entering revenue
  service (first car June 2026). Match-day operations in June-July
  will use legacy 6-car consists. The GT study's empirical 707-pax
  figure is the best benchmark for the fleet actually running; CQ400
  seated/service/crush are a modeling framework.
- **Post-game egress.** Historically the harder problem (22,000 riders
  overwhelmed a 15,000 plan at the 2018 CFP Championship). MARTA's
  2019 fix stood up to Super Bowl LIII. Not modeled here.
- **Sustained tournament demand.** 300,000+ cumulative visitors across
  the month, Fan Fest crowds, airport surge, non-stadium tourism. Not
  modeled.

These exclusions are intentional. The model is simple enough to build
in an afternoon and still produces a defensible answer.


## 5. Save and preview Day 5


In [ ]:
assert (CHECKPOINTS / 'day4_output_demand_estimates.csv').exists()
n_s = (gap_df['verdict'] == 'SURPLUS').sum()
n_d = (gap_df['verdict'] == 'DEFICIT').sum()
print(f'✅ Day 4 done. Gap grid: {n_s}/9 surplus, {n_d}/9 deficit.')
print(f'   Tipping point: high mode share × service standard.')
print(f'   Tomorrow you write the briefing.')
